In [52]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [53]:
import os
os.chdir(r'C:\Users\Lenovo\Desktop\Diabetes_Classifier\notebooks')
os.chdir("..")
print(os.getcwd())

C:\Users\Lenovo\Desktop\Diabetes_Classifier


In [54]:
from modeling.XGBoost import XGBoost
from modeling.RandomForest import RandomForest
from modeling.KNN import KNN
from modeling.MLP import MLP
from modeling.Ensemble import LR
from src.data_feature_engineering import features
from src.data_preprocessing import scaling
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score,recall_score,precision_score,balanced_accuracy_score,roc_auc_score,accuracy_score
import warnings
warnings.filterwarnings('ignore')

In [55]:
PATH="data/clean/clean2.csv"
df=pd.read_csv(PATH)
df.head()
df.columns = df.columns.str.strip()

In [56]:


y=df['diagnosis']
X=df.drop(['diagnosis'],axis=1)

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
X_train,X_test=features(X_train,X_test)


In [57]:

xgb=XGBoost()
xgb_params={'n_estimators': 7800, 'max_depth': 13, 'learning_rate': 0.03585979413065538, 'gamma': 0.00025046646096832056, 'min_child_weight': 8, 'reg_alpha': 9.979812055651985, 'reg_lambda': 0.041477402642739976, 'subsample': 0.9999355054935987, 'colsample_bytree': 0.8896668801909641, 'tree_method': 'hist', 'n_jobs': -1}
xgb.set_params(xgb_params)
pd.set_option('display.max_rows',None)

#Optuna
# params=xgb.hyperparameter_tuning(X,y,30)
# xgb.set_params(params)



In [58]:
rf=RandomForest()
rf_params={'n_estimators': 600, 'max_depth': 10, 'min_samples_split': 12, 'min_samples_leaf': 7, 'max_features': 'log2', 'bootstrap': True, 'criterion': 'log_loss', 'class_weight': None, 'n_jobs': -1, 'random_state': 42}
rf.set_params(rf_params)

# rf_params=rf.hyperparameter_tuning(X_train,y_train,20)


In [59]:
mlp=MLP()
mlp_params={'hidden_layer_sizes': (32,), 'activation': 'relu', 'solver': 'adam', 'alpha': 0.00017146311867992375, 'learning_rate_init': 0.004552010146185333, 'batch_size': 64, 'max_iter': 400, 'early_stopping': True, 'random_state': 42}

mlp.set_params(mlp_params)

X_train_scaled,X_test_scaled=scaling(X_train,X_test,'anomaly_score')

# params=mlp.hyperparameter_tuning(X_train_scaled,y_train,20)
# mlp.set_params(params)



In [60]:
knn=KNN()
knn_params={'n_neighbors': 28, 'weights': 'uniform', 'metric': 'minkowski', 'p': 2, 'algorithm': 'kd_tree', 'leaf_size': 12, 'n_jobs': -1}
knn.set_params(knn_params)

# parameters=knn.hyperparameter_tuning(X_train_scaled,y_train,20)

In [61]:
ensemble=[xgb,rf,mlp,knn]
X_train_copy=X_train_scaled.copy()
X_test_copy=X_test_scaled.copy()
oof=pd.DataFrame()
oof_test=pd.DataFrame()
for model in ensemble:
    X_train_scaled[f'OOF_{type(model).__name__}'],X_test_scaled[f'OOF_{type(model).__name__}']=model.oof(X_train,y_train,X_test,type(model).__name__,10)

XGBoost fold 1/10 done
XGBoost fold 2/10 done
XGBoost fold 3/10 done
XGBoost fold 4/10 done
XGBoost fold 5/10 done
XGBoost fold 6/10 done
XGBoost fold 7/10 done
XGBoost fold 8/10 done
XGBoost fold 9/10 done
XGBoost fold 10/10 done
RandomForest fold 1/10 done
RandomForest fold 2/10 done
RandomForest fold 3/10 done
RandomForest fold 4/10 done
RandomForest fold 5/10 done
RandomForest fold 6/10 done
RandomForest fold 7/10 done
RandomForest fold 8/10 done
RandomForest fold 9/10 done
RandomForest fold 10/10 done
MLP fold 1/10 done
MLP fold 2/10 done
MLP fold 3/10 done
MLP fold 4/10 done
MLP fold 5/10 done
MLP fold 6/10 done
MLP fold 7/10 done
MLP fold 8/10 done
MLP fold 9/10 done
MLP fold 10/10 done
KNN fold 1/10 done
KNN fold 2/10 done
KNN fold 3/10 done
KNN fold 4/10 done
KNN fold 5/10 done
KNN fold 6/10 done
KNN fold 7/10 done
KNN fold 8/10 done
KNN fold 9/10 done
KNN fold 10/10 done


In [62]:

oof.tail(2)

""


In [63]:
meta_x_model=LR()
params={'penalty':'l2',
    'C':0.01,
    'solver':'lbfgs',
    'max_iter':5000,
    'class_weight':'balanced',   # or None
    'random_state':42,
    'n_jobs':-1}

meta_x_model.set_params(params)

In [64]:
# X_eval=pd.concat([X_train_scaled,X_test_scaled])
# y_eval=pd.concat([y_train,y_test])


In [65]:
def compare(base: dict, meta: dict):
    for key in base:
        diff = meta[key] - base[key]
        symbol = "↑" if diff > 0 else ("↓" if diff < 0 else "=")
        print(f"{key}: base={base[key]:.4f} meta={meta[key]:.4f} {symbol} ({diff:+.4f})")

In [66]:
# for model in ensemble:
#     print("=================================")
#     print(f"Model: {model}")
#     if model.__class__.__name__ in ["XGBoost", "RandomForest"]:
        
#         compare(model.evaluation(X,y,5).values(),meta_x_model.evaluation(X_eval,y_eval).values())
#     else:
#         compare(model.evaluation(X_eval,y_eval,5).values(),meta_x_model.evaluation(X_eval,y_eval).values())

In [67]:
meta_x_model.train(X_train_scaled,y_train)
lr_proba=meta_x_model.predict_proba(X_test_scaled)
preds=meta_x_model.predict(X_test_scaled)


In [ ]:

def measure(y_pred: np.ndarray, y_proba: np.ndarray, y_real: np.ndarray) -> dict:
    metrics = {}
    metrics['accuracy'] = accuracy_score(y_real, y_pred)
    metrics['balanced']=balanced_accuracy_score(y_real,y_pred)
    metrics['precision'] = precision_score(y_real, y_pred)
    metrics['f1'] = f1_score(y_real, y_pred)
    metrics['recall'] = recall_score(y_real, y_pred)
    metrics['roc_auc'] = roc_auc_score(y_real, y_proba[:,1])
    return metrics

    

In [69]:
logistic_regression_metrics=measure(preds,lr_proba,y_test)
print(logistic_regression_metrics)
# {'accuracy': 0.8383641674780915, 'balanced': 0.8406224152191895, 'precision': 0.8300567967262891, 'f1': 0.8335305789255778, 'recall': 0.8406224152191895, 'roc_auc': 0.9163286568683591}

{'accuracy': 0.8169425511197663, 'balanced': 0.814616504626368, 'precision': 0.8107007019388355, 'f1': 0.8123303862333886, 'recall': 0.814616504626368, 'roc_auc': 0.900765973102876}


In [70]:
for model in ensemble:
    if model.__class__.__name__ in ["XGBoost", "RandomForest"]:
        model.train(X_train,y_train)
        print(model.__class__.__name__)
    else:
        model.train(X_train_copy,y_train)

XGBoost
RandomForest


In [71]:
for model in ensemble:
    print(f'Model:{model}')
    if model.__class__.__name__ in ["XGBoost", "RandomForest"]:
        predictions=model.predict(X_test)
        proba=model.predict_proba(X_test)
        vals=measure(predictions,proba,y_test)
        compare(vals,logistic_regression_metrics)
    else:
        predictions=model.predict(X_test_copy)
        proba=model.predict_proba(X_test_copy)
        vals=measure(predictions,proba,y_test)
        compare(vals,logistic_regression_metrics)

Model:<modeling.XGBoost.XGBoost object at 0x0000024F34229D10>
accuracy: base=0.8150 meta=0.8169 ↑ (+0.0019)
balanced: base=0.8002 meta=0.8146 ↑ (+0.0144)
precision: base=0.8144 meta=0.8107 ↓ (-0.0037)
f1: base=0.8051 meta=0.8123 ↑ (+0.0072)
recall: base=0.8002 meta=0.8146 ↑ (+0.0144)
roc_auc: base=0.9061 meta=0.9008 ↓ (-0.0053)
Model:<modeling.RandomForest.RandomForest object at 0x0000024F3435A210>


accuracy: base=0.8189 meta=0.8169 ↓ (-0.0019)
balanced: base=0.8032 meta=0.8146 ↑ (+0.0115)
precision: base=0.8198 meta=0.8107 ↓ (-0.0091)
f1: base=0.8087 meta=0.8123 ↑ (+0.0036)
recall: base=0.8032 meta=0.8146 ↑ (+0.0115)
roc_auc: base=0.9076 meta=0.9008 ↓ (-0.0068)
Model:<modeling.MLP.MLP object at 0x0000024F3435A0D0>
accuracy: base=0.8101 meta=0.8169 ↑ (+0.0068)
balanced: base=0.8007 meta=0.8146 ↑ (+0.0139)
precision: base=0.8052 meta=0.8107 ↑ (+0.0055)
f1: base=0.8026 meta=0.8123 ↑ (+0.0097)
recall: base=0.8007 meta=0.8146 ↑ (+0.0139)
roc_auc: base=0.9027 meta=0.9008 ↓ (-0.0019)
Model:<modeling.KNN.KNN object at 0x0000024F34358A50>
accuracy: base=0.7965 meta=0.8169 ↑ (+0.0204)
balanced: base=0.7760 meta=0.8146 ↑ (+0.0386)
precision: base=0.8001 meta=0.8107 ↑ (+0.0106)
f1: base=0.7826 meta=0.8123 ↑ (+0.0298)
recall: base=0.7760 meta=0.8146 ↑ (+0.0386)
roc_auc: base=0.8931 meta=0.9008 ↑ (+0.0077)


In [72]:
X_train_scaled.iloc[:,-4:].corr()

,OOF_XGBoost,OOF_RandomForest,OOF_MLP,OOF_KNN
OOF_XGBoost,1.000000,0.984821,0.908257,0.842542
OOF_RandomForest,0.984821,1.000000,0.912504,0.845743
OOF_MLP,0.908257,0.912504,1.000000,0.862553
OOF_KNN,0.842542,0.845743,0.862553,1.000000


In [73]:
print(oof)

Empty DataFrame
Columns: []
Index: []


In [74]:
feature_cols=X_train_scaled.columns
meta_x_model.feature_importance(feature_cols)

             feature  coefficient
26  OOF_RandomForest     0.551880
25       OOF_XGBoost     0.522231
0                age     0.469468
9          Age_x_BMI     0.462255
20         Cluster 4     0.283747
5                ldl     0.283119
6                 cr    -0.239139
1                bmi     0.237725
28           OOF_KNN     0.189135
3                 tg     0.165158
21         Cluster 5     0.151311
16         Cluster 0     0.148796
2               chol    -0.144968
27           OOF_MLP     0.140205
15    cluster_labels    -0.123123
4                hdl    -0.102847
13          bun_x_cr    -0.100018
7                bun     0.076931
14          chol/ldl    -0.065138
8             lipids    -0.064575
23   anomaly_score_1    -0.057030
18         Cluster 2     0.044112
24   anomaly_score_2     0.038248
10         HDL_x_LDL     0.038092
19         Cluster 3     0.034263
11           BMI/LDL     0.028640
17         Cluster 1    -0.015493
12       BMI/HDL+LDL    -0.013663
22          ge